# 📥 Ingestion: Kaggle API Data Extraction

## 📑 Notebook Information
| Version | Date | Author | Summary of Changes |
| :--- | :--- | :--- | :--- |
| 1.0 | 2024-05-20 | Tássia Marchito | Initial script for Kaggle API integration. |
| 1.1 | 2024-05-22 | Tássia Marchito | Translated metadata and refined volume comments to English. |

### 📊 Notebook Overview
This notebook automates the data acquisition process by connecting to the **Kaggle API** and downloading the Olist e-commerce dataset directly into the Bronze layer's landing zone.

### 🎯 Objectives
- **API Authentication:** Securely connect to Kaggle using environment credentials.
- **Data Ingestion:** Download and extract raw CSV files into the Unity Catalog Volume.
- **Volume Governance:** Apply technical descriptions to the `raw_files` volume for documentation.

---

In [0]:
%sql
-- 1. Updating Volume Metadata
-- Documenting the landing zone for raw data governance

COMMENT ON VOLUME cat_tm_services_bronze.db_logistics.raw_files IS 'Landing zone for CSV files downloaded from Kaggle API.';

In [0]:
# 0. Install the Kaggle API Client
%pip install kaggle --upgrade

import os
import zipfile

# 1. Secure Credential Retrieval from Secrets Manager
try:
    os.environ['KAGGLE_USERNAME'] = dbutils.secrets.get(scope="logistics_credentials", key="kaggle_user")
    os.environ['KAGGLE_KEY'] = dbutils.secrets.get(scope="logistics_credentials", key="kaggle_key")
    print("✅ Kaggle credentials loaded securely from Secret Manager.")
except Exception as e:
    print(f"❌ Error loading secrets: {e}. Please ensure the scope and keys exist.")

# 2. Path Configuration
dataset = "olistbr/brazilian-ecommerce"
volume_path = "/Volumes/cat_tm_services_bronze/db_logistics/raw_files"

# 3. Automated Data Ingestion
print(f"Starting ingestion for dataset: {dataset}")
# The !kaggle command will now be recognized after the %pip install
!kaggle datasets download -d {dataset} -p {volume_path}

# 4. Extraction and File Cleanup
zip_file = f"{volume_path}/brazilian-ecommerce.zip"
if os.path.exists(zip_file):
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(volume_path)
    os.remove(zip_file)
    print(f"✅ Extraction complete. Files available at: {volume_path}")
else:
    print(f"❌ Zip file not found at {zip_file}. Check the download step logs.")

In [0]:
# --- Ingestion Quality Assurance (QA Gate) ---
# Objective: Verify if all raw files were successfully extracted and stored in the Volume.

from pyspark.sql.functions import col

# 1. Configuration
raw_path = "/Volumes/cat_tm_services_bronze/db_logistics/raw_files"
expected_files_count = 9 # Adjust based on your Kaggle dataset

print(f"🧐 Validating Raw Extraction at: {raw_path}...")

try:
    # 2. Extract metadata from the landing zone
    files = dbutils.fs.ls(raw_path)
    df_files = spark.createDataFrame(files) \
        .select(
            col("name").alias("file_name"), 
            (col("size") / 1024).alias("size_kb"),
            col("path")
        )

    # 3. Validation Metrics
    actual_files_count = df_files.count()
    total_size_kb = df_files.agg({"size_kb": "sum"}).collect()[0][0]
    empty_files = df_files.filter(col("size_kb") == 0).count()

    print(f"\n📊 Extraction Metrics:")
    print(f"  - Expected Files: {expected_files_count}")
    print(f"  - Actual Files: {actual_files_count}")
    print(f"  - Total Ingested Volume: {total_size_kb:.2f} KB")
    print(f"  - Empty Files Detected: {empty_files}")

    # 4. Final Verdict
    if actual_files_count >= expected_files_count and empty_files == 0:
        print("\n✅ QA STATUS: SUCCESS")
        print("Raw files are present and ready for Bronze processing.")
    else:
        print("\n⚠️ QA STATUS: WARNING")
        print("Check if all Kaggle files were downloaded correctly or if there are empty assets.")

    # 5. Visual Check
    display(df_files.sort("size_kb", ascending=False))

except Exception as e:
    print(f"\n❌ QA STATUS: FAILED")
    print(f"Directory not found or access denied. Error: {e}")